***Summary***

**GA with NBC**
         
*AE*

- Runtime: 1 minute
- SVM Acc: 97.75 %
- NBC Acc: 92.13 %
- RF Acc:  94.38 %

*VAE*

- Runtime: 1 minute
- SVM Acc: 97.75 %
- NBC Acc: 98.87 %
- RF Acc:  96.63 %

**GA with SVM**

*AE*

- Runtime: 3 minutes
- SVM Acc: 95.5 %
- NBC Acc: 97.75 %
- RF Acc:  96.63 %

*VAE* 

- Runtime: 3 minutes
- SVM Acc: 95.5 %
- NBC Acc: 95.5 %
- RF Acc:  94.38 %

**Training GA with NBC**

**AE + GA**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx') 
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model

class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        #Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh() 
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance losses
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32


# Initialize model, optimizer, and scheduler
model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train).numpy()
    test_latent = model.encoder(X_test).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

from deap import base, creator, tools, algorithms
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
import random
import warnings
warnings.filterwarnings("ignore")

print("Starting GA")
print("latent data:", train_latent.shape)

# Define all the needed parameters

LATENT_FEATURES = train_latent.shape[1]
N_GENERATIONS = 50
POPULATION_SIZE = 100
P_CROSSOVER = 0.7
P_MUTATION = 0.2
TOURNAMENT_SIZE = 3
N_SELECTED_LATENT_FEATURES = 10

try:
    del creator.FitnessMax
    del creator.Individual
except:
    pass

# Create types for GA
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=LATENT_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate_individual_with_latent(individual):
    # Get selected latent features
    selected_indices = [i for i, val in enumerate(individual) if val == 1]
    
    # If too few features are selected, penalize the individual
    if len(selected_indices) < 2:
        return 0.0,
    
    # Create a modified input tensor with zeros for non-selected features
    modified_latent = train_latent[:, selected_indices]
    
    # Train a simple classifier on the selected latent features
    classifier = GaussianNB()
    scores = cross_val_score(classifier, modified_latent, y_train_np, cv=5)
    accuracy = scores.mean()

    return accuracy,

toolbox.register("evaluate", evaluate_individual_with_latent)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

def run_ga():
    # Initialize population
    population = toolbox.population(n=POPULATION_SIZE)
    
    # Statistics to track progress
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Hall of fame to keep track of best individuals
    hof = tools.HallOfFame(1)
    
    # Running the algorithm
    population, logbook = algorithms.eaSimple(
        population, 
        toolbox, 
        cxpb=P_CROSSOVER, 
        mutpb=P_MUTATION, 
        ngen=N_GENERATIONS, 
        stats=stats, 
        halloffame=hof, 
        verbose=True
    )
    
    return population, logbook, hof

population, logbook, hof = run_ga()

# Get the best individual
best_individual = hof[0]
selected_latent_features = [i for i, val in enumerate(best_individual) if val == 1]
num_selected = len(selected_latent_features)

print(f"Number of selected latent features: {num_selected} out of {LATENT_FEATURES}")
print('The selected features: ', selected_latent_features)
print(f"Best fitness: {best_individual.fitness.values[0]}")


#testing the ga selected features on the models
print('Testing GA Selected Features: ')
modified_latent_train = train_latent[:, selected_latent_features]
modified_latent_test = test_latent[:, selected_latent_features]

# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(modified_latent_train, y_train_np)
svm_preds = svm_model.predict(modified_latent_test)

svm_acc = accuracy_score(y_test_np, svm_preds)
svm_percision = precision_score(y_test_np, svm_preds, average='macro')
svm_recall = recall_score(y_test_np, svm_preds, average='macro')
svm_f1 = f1_score(y_test_np, svm_preds, average='macro')

print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")
print(f"SVM Percision on latent features selected by GA: {svm_percision * 100:.4f}")
print(f"SVM Recall on latent features selected by GA: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on latent features selected by GA: {svm_f1 * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(modified_latent_train, y_train_np)
nbc_preds = nbc_model.predict(modified_latent_test)

nbc_acc = accuracy_score(y_test_np, nbc_preds)
nbc_percision = precision_score(y_test_np, nbc_preds, average='macro')
nbc_recall = recall_score(y_test_np, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test_np, nbc_preds, average='macro')

print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on latent features selected by GA: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on latent features selected by GA: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on latent features selected by GA: {nbc_f1 * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(modified_latent_train, y_train_np)
rf_preds = rf_model.predict(modified_latent_test)

rf_acc = accuracy_score(y_test_np, rf_preds)
rf_percision = precision_score(y_test_np, rf_preds, average='macro')
rf_recall = recall_score(y_test_np, rf_preds, average='macro')
rf_f1 = f1_score(y_test_np, rf_preds, average='macro')

print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by GA: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by GA: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by GA: {rf_f1 * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.1308
Epoch [2/50], Loss: 0.7803
Epoch [3/50], Loss: 0.5798
Epoch [4/50], Loss: 0.5677
Epoch [5/50], Loss: 0.4777
Epoch [6/50], Loss: 0.4631
Epoch [7/50], Loss: 0.5088
Epoch [8/50], Loss: 0.4200
Epoch [9/50], Loss: 0.4285
Epoch [10/50], Loss: 0.4108
Epoch [11/50], Loss: 0.4223
Epoch [12/50], Loss: 0.3706
Epoch [13/50], Loss: 0.3960
Epoch [14/50], Loss: 0.4138
Epoch [15/50], Loss: 0.4115
Epoch [16/50], Loss: 0.3978
Epoch [17/50], Loss: 0.4193
Epoch [18/50], Loss: 0.3936
Epoch [19/50], Loss: 0.4839
Epoch [20/50], Loss: 0.4359
Epoch [21/50], Loss: 0.3559
Epoch [22/50], Loss: 0.3978
Epoch [23/50], Loss: 0.3493
Epoch [24/50], Loss: 0.3858
Epoch [25/50], Loss: 0.3962
Epoch [26/50], Loss: 0.3818
Epoch [27/50], Loss: 0.3643
Epoch [28/50], Loss: 0.3734
Epoch [29/50], Loss: 0.3715
Epoch [30/50], Loss: 0.4355
Epoch [31/50], Loss: 0.4068
Epoch [32/50], Loss: 0.4249
Epoch [33/50], Loss: 0.3807
Epoch [34/50], Loss: 0.3824
Epoch [35/50], Loss: 0.3796


**VAE + GA**

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx')
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Variational Autoencoder-Classifier Model

class JointVAEClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointVAEClassifier, self).__init__()
        # Encoder
        self.encoder_net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128)
        )
        # Layers to produce the mean and log-variance for the latent distribution
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        # Encode input to get intermediate representation
        x_encoded = self.encoder_net(x)
        # Produce latent distribution parameters
        mu = self.fc_mu(x_encoded)
        logvar = self.fc_logvar(x_encoded)
        # Sample latent vector
        z = self.reparameterize(mu, logvar)
        # Reconstruct input and classify based on latent vector
        reconstruction = self.decoder(z)
        logits = self.classifier(z)
        return reconstruction, logits, mu, logvar


# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance reconstruction and KL loss and classification loss
lambda_recon = 0.5

def vae_combined_loss(reconstructed, original, logits, labels, mu, logvar):
    # Reconstruction loss
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    # KL divergence loss: average over batch
    loss_kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    # Classification loss
    loss_class = classification_loss_fn(logits, labels)
    # Combined loss: balance reconstruction (with KL) and classification
    return lambda_recon * (loss_recon + loss_kl) + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

# Initialize model, optimizer, and scheduler
model = JointVAEClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50


# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits, mu, logvar = model(inputs)
        loss = vae_combined_loss(reconstruction, inputs, logits, labels, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")


# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_encoded = model.encoder_net(X_train)
    train_latent = model.fc_mu(train_encoded).numpy()
    
    test_encoded = model.encoder_net(X_test)
    test_latent = model.fc_mu(test_encoded).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

# --- Genetic Algorithm Implementation ---

from deap import base, creator, tools, algorithms
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
import random
import warnings
warnings.filterwarnings("ignore")

print("Starting GA")
print("latent data:", train_latent.shape)

# Define all the needed parameters

LATENT_FEATURES = train_latent.shape[1]
N_GENERATIONS = 50
POPULATION_SIZE = 100
P_CROSSOVER = 0.7
P_MUTATION = 0.2
TOURNAMENT_SIZE = 3
N_SELECTED_LATENT_FEATURES = 10

try:
    del creator.FitnessMax
    del creator.Individual
except:
    pass

# Create types for GA
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=LATENT_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate_individual_with_latent(individual):
    # Get selected latent features
    selected_indices = [i for i, val in enumerate(individual) if val == 1]
    
    # If too few features are selected, penalize the individual
    if len(selected_indices) < 2:
        return 0.0,
    
    # Create a modified input tensor with zeros for non-selected features
    modified_latent = train_latent[:, selected_indices]
    
    # Train a simple classifier on the selected latent features
    classifier = GaussianNB()
    scores = cross_val_score(classifier, modified_latent, y_train_np, cv=5)
    accuracy = scores.mean()

    return accuracy,

toolbox.register("evaluate", evaluate_individual_with_latent)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

def run_ga():
    # Initialize population
    population = toolbox.population(n=POPULATION_SIZE)
    
    # Statistics to track progress
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Hall of fame to keep track of best individuals
    hof = tools.HallOfFame(1)
    
    # Running the algorithm
    population, logbook = algorithms.eaSimple(
        population, 
        toolbox, 
        cxpb=P_CROSSOVER, 
        mutpb=P_MUTATION, 
        ngen=N_GENERATIONS, 
        stats=stats, 
        halloffame=hof, 
        verbose=True
    )
    
    return population, logbook, hof

population, logbook, hof = run_ga()

# Get the best individual
best_individual = hof[0]
selected_latent_features = [i for i, val in enumerate(best_individual) if val == 1]
num_selected = len(selected_latent_features)

print(f"Number of selected latent features: {num_selected} out of {LATENT_FEATURES}")
print('The selected features: ', selected_latent_features)
print(f"Best fitness: {best_individual.fitness.values[0]}")


#testing the ga selected features on the models
print('Testing GA Selected Features: ')
modified_latent_train = train_latent[:, selected_latent_features]
modified_latent_test = test_latent[:, selected_latent_features]

# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(modified_latent_train, y_train_np)
svm_preds = svm_model.predict(modified_latent_test)

svm_acc = accuracy_score(y_test_np, svm_preds)
svm_percision = precision_score(y_test_np, svm_preds, average='macro')
svm_recall = recall_score(y_test_np, svm_preds, average='macro')
svm_f1 = f1_score(y_test_np, svm_preds, average='macro')

print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")
print(f"SVM Percision on latent features selected by GA: {svm_percision * 100:.4f}")
print(f"SVM Recall on latent features selected by GA: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on latent features selected by GA: {svm_f1 * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(modified_latent_train, y_train_np)
nbc_preds = nbc_model.predict(modified_latent_test)

nbc_acc = accuracy_score(y_test_np, nbc_preds)
nbc_percision = precision_score(y_test_np, nbc_preds, average='macro')
nbc_recall = recall_score(y_test_np, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test_np, nbc_preds, average='macro')

print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on latent features selected by GA: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on latent features selected by GA: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on latent features selected by GA: {nbc_f1 * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(modified_latent_train, y_train_np)
rf_preds = rf_model.predict(modified_latent_test)

rf_acc = accuracy_score(y_test_np, rf_preds)
rf_percision = precision_score(y_test_np, rf_preds, average='macro')
rf_recall = recall_score(y_test_np, rf_preds, average='macro')
rf_f1 = f1_score(y_test_np, rf_preds, average='macro')

print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by GA: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by GA: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by GA: {rf_f1 * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.4971
Epoch [2/50], Loss: 1.3046
Epoch [3/50], Loss: 1.1806
Epoch [4/50], Loss: 1.0843
Epoch [5/50], Loss: 1.0307
Epoch [6/50], Loss: 0.9337
Epoch [7/50], Loss: 0.8673
Epoch [8/50], Loss: 0.8817
Epoch [9/50], Loss: 0.8451
Epoch [10/50], Loss: 0.8027
Epoch [11/50], Loss: 0.7682
Epoch [12/50], Loss: 0.7529
Epoch [13/50], Loss: 0.8029
Epoch [14/50], Loss: 0.7132
Epoch [15/50], Loss: 0.7105
Epoch [16/50], Loss: 0.7166
Epoch [17/50], Loss: 0.7352
Epoch [18/50], Loss: 0.7178
Epoch [19/50], Loss: 0.7454
Epoch [20/50], Loss: 0.7035
Epoch [21/50], Loss: 0.6985
Epoch [22/50], Loss: 0.7129
Epoch [23/50], Loss: 0.6663
Epoch [24/50], Loss: 0.6580
Epoch [25/50], Loss: 0.7012
Epoch [26/50], Loss: 0.7431
Epoch [27/50], Loss: 0.7077
Epoch [28/50], Loss: 0.7407
Epoch [29/50], Loss: 0.7084
Epoch [30/50], Loss: 0.6981
Epoch [31/50], Loss: 0.6980
Epoch [32/50], Loss: 0.6702
Epoch [33/50], Loss: 0.6797
Epoch [34/50], Loss: 0.6607
Epoch [35/50], Loss: 0.6269


**Training GA with RF**

**AE + GA**

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx') 
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model

class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        #Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh() 
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance losses
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32


# Initialize model, optimizer, and scheduler
model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train).numpy()
    test_latent = model.encoder(X_test).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

from deap import base, creator, tools, algorithms
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import random
import warnings
warnings.filterwarnings("ignore")

print("Starting GA")
print("latent data:", train_latent.shape)

# Define all the needed parameters

LATENT_FEATURES = train_latent.shape[1]
N_GENERATIONS = 50
POPULATION_SIZE = 100
P_CROSSOVER = 0.7
P_MUTATION = 0.2
TOURNAMENT_SIZE = 3
N_SELECTED_LATENT_FEATURES = 10

try:
    del creator.FitnessMax
    del creator.Individual
except:
    pass

# Create types for GA
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=LATENT_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate_individual_with_latent(individual):
    # Get selected latent features
    selected_indices = [i for i, val in enumerate(individual) if val == 1]
    
    # If too few features are selected, penalize the individual
    if len(selected_indices) < 2:
        return 0.0,
    
    # Create a modified input tensor with zeros for non-selected features
    modified_latent = train_latent[:, selected_indices]
    
    # Train a simple classifier on the selected latent features
    classifier = RandomForestClassifier(random_state=42)
    scores = cross_val_score(classifier, modified_latent, y_train_np, cv=5)
    accuracy = scores.mean()

    return accuracy,

toolbox.register("evaluate", evaluate_individual_with_latent)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

def run_ga():
    # Initialize population
    population = toolbox.population(n=POPULATION_SIZE)
    
    # Statistics to track progress
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Hall of fame to keep track of best individuals
    hof = tools.HallOfFame(1)
    
    # Running the algorithm
    population, logbook = algorithms.eaSimple(
        population, 
        toolbox, 
        cxpb=P_CROSSOVER, 
        mutpb=P_MUTATION, 
        ngen=N_GENERATIONS, 
        stats=stats, 
        halloffame=hof, 
        verbose=True
    )
    
    return population, logbook, hof


population, logbook, hof = run_ga()

# Get the best individual
best_individual = hof[0]
selected_latent_features = [i for i, val in enumerate(best_individual) if val == 1]
num_selected = len(selected_latent_features)

print(f"Number of selected latent features: {num_selected} out of {LATENT_FEATURES}")
print('The selected features: ', selected_latent_features)
print(f"Best fitness: {best_individual.fitness.values[0]}")


#testing the ga selected features on the models
print('Testing GA Selected Features: ')
modified_latent_train = train_latent[:, selected_latent_features]
modified_latent_test = test_latent[:, selected_latent_features]

# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(modified_latent_train, y_train_np)
svm_preds = svm_model.predict(modified_latent_test)
svm_acc = accuracy_score(y_test_np, svm_preds)
print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(modified_latent_train, y_train_np)
nbc_preds = nbc_model.predict(modified_latent_test)
nbc_acc = accuracy_score(y_test_np, nbc_preds)
print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(modified_latent_train, y_train_np)
rf_preds = rf_model.predict(modified_latent_test)
rf_acc = accuracy_score(y_test_np, rf_preds)
print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.1457
Epoch [2/50], Loss: 0.7852
Epoch [3/50], Loss: 0.6541
Epoch [4/50], Loss: 0.5612
Epoch [5/50], Loss: 0.5355
Epoch [6/50], Loss: 0.4839
Epoch [7/50], Loss: 0.5014
Epoch [8/50], Loss: 0.4969
Epoch [9/50], Loss: 0.4146
Epoch [10/50], Loss: 0.4235
Epoch [11/50], Loss: 0.4362
Epoch [12/50], Loss: 0.3803
Epoch [13/50], Loss: 0.3994
Epoch [14/50], Loss: 0.3800
Epoch [15/50], Loss: 0.4324
Epoch [16/50], Loss: 0.4045
Epoch [17/50], Loss: 0.4386
Epoch [18/50], Loss: 0.3696
Epoch [19/50], Loss: 0.4029
Epoch [20/50], Loss: 0.4034
Epoch [21/50], Loss: 0.4335
Epoch [22/50], Loss: 0.3633
Epoch [23/50], Loss: 0.3749
Epoch [24/50], Loss: 0.4007
Epoch [25/50], Loss: 0.4182
Epoch [26/50], Loss: 0.4454
Epoch [27/50], Loss: 0.4339
Epoch [28/50], Loss: 0.3804
Epoch [29/50], Loss: 0.3592
Epoch [30/50], Loss: 0.3618
Epoch [31/50], Loss: 0.3886
Epoch [32/50], Loss: 0.3893
Epoch [33/50], Loss: 0.4357
Epoch [34/50], Loss: 0.3695
Epoch [35/50], Loss: 0.3820


KeyboardInterrupt: 

**VAE + GA**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx')
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Variational Autoencoder-Classifier Model

class JointVAEClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointVAEClassifier, self).__init__()
        # Encoder
        self.encoder_net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128)
        )
        # Layers to produce the mean and log-variance for the latent distribution
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        # Encode input to get intermediate representation
        x_encoded = self.encoder_net(x)
        # Produce latent distribution parameters
        mu = self.fc_mu(x_encoded)
        logvar = self.fc_logvar(x_encoded)
        # Sample latent vector
        z = self.reparameterize(mu, logvar)
        # Reconstruct input and classify based on latent vector
        reconstruction = self.decoder(z)
        logits = self.classifier(z)
        return reconstruction, logits, mu, logvar


# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance reconstruction and KL loss and classification loss
lambda_recon = 0.5

def vae_combined_loss(reconstructed, original, logits, labels, mu, logvar):
    # Reconstruction loss
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    # KL divergence loss: average over batch
    loss_kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    # Classification loss
    loss_class = classification_loss_fn(logits, labels)
    # Combined loss: balance reconstruction (with KL) and classification
    return lambda_recon * (loss_recon + loss_kl) + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

# Initialize model, optimizer, and scheduler
model = JointVAEClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50


# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits, mu, logvar = model(inputs)
        loss = vae_combined_loss(reconstruction, inputs, logits, labels, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")


# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_encoded = model.encoder_net(X_train)
    train_latent = model.fc_mu(train_encoded).numpy()
    
    test_encoded = model.encoder_net(X_test)
    test_latent = model.fc_mu(test_encoded).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

# --- Genetic Algorithm Implementation ---

import random
from deap import base, creator, tools, algorithms

print("Starting GA")
print("latent data:", train_latent.shape)

# Define all the needed parameters

LATENT_FEATURES = train_latent.shape[1]
N_GENERATIONS = 50
POPULATION_SIZE = 100
P_CROSSOVER = 0.7
P_MUTATION = 0.2
TOURNAMENT_SIZE = 3
N_SELECTED_LATENT_FEATURES = 10

try:
    del creator.FitnessMax
    del creator.Individual
except:
    pass

# Create types for GA
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=LATENT_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate_individual_with_latent(individual):
    # Get selected latent features
    selected_indices = [i for i, val in enumerate(individual) if val == 1]
    
    # If too few features are selected, penalize the individual
    if len(selected_indices) < 2:
        return 0.0,
    
    # Create a modified input tensor with zeros for non-selected features
    modified_latent = train_latent[:, selected_indices]
    
    # Train a simple classifier on the selected latent features
    classifier = RandomForestClassifier(random_state=42)
    scores = cross_val_score(classifier, modified_latent, y_train_np, cv=5)
    accuracy = scores.mean()

    return accuracy,

toolbox.register("evaluate", evaluate_individual_with_latent)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

def run_ga():
    # Initialize population
    population = toolbox.population(n=POPULATION_SIZE)
    
    # Statistics to track progress
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Hall of fame to keep track of best individuals
    hof = tools.HallOfFame(1)
    
    # Running the algorithm
    population, logbook = algorithms.eaSimple(
        population, 
        toolbox, 
        cxpb=P_CROSSOVER, 
        mutpb=P_MUTATION, 
        ngen=N_GENERATIONS, 
        stats=stats, 
        halloffame=hof, 
        verbose=True
    )
    
    return population, logbook, hof

'''import time
print("Starting Genetic Algorithm for latent feature selection")
start_time = time.time()
population, logbook, hof = run_ga()
end_time = time.time()
print(f"GA completed in {(end_time - start_time)/60:.2f} minutes")'''

population, logbook, hof = run_ga()

# Get the best individual
best_individual = hof[0]
selected_latent_features = [i for i, val in enumerate(best_individual) if val == 1]
num_selected = len(selected_latent_features)

print(f"Number of selected latent features: {num_selected} out of {LATENT_FEATURES}")
print('The selected features: ', selected_latent_features)
print(f"Best fitness: {best_individual.fitness.values[0]}")


#testing the ga selected features on the models
print('Testing GA Selected Features: ')
modified_latent_train = train_latent[:, selected_latent_features]
modified_latent_test = test_latent[:, selected_latent_features]

# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(modified_latent_train, y_train_np)
svm_preds = svm_model.predict(modified_latent_test)
svm_acc = accuracy_score(y_test_np, svm_preds)
print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(modified_latent_train, y_train_np)
nbc_preds = nbc_model.predict(modified_latent_test)
nbc_acc = accuracy_score(y_test_np, nbc_preds)
print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(modified_latent_train, y_train_np)
rf_preds = rf_model.predict(modified_latent_test)
rf_acc = accuracy_score(y_test_np, rf_preds)
print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")

**Training GA with SVM**

**AE + GA**

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx') 
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model

class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        #Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh() 
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance losses
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32


# Initialize model, optimizer, and scheduler
model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train).numpy()
    test_latent = model.encoder(X_test).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

from deap import base, creator, tools, algorithms
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
import random
import warnings
warnings.filterwarnings("ignore")

print("Starting GA")
print("latent data:", train_latent.shape)

# Define all the needed parameters

LATENT_FEATURES = train_latent.shape[1]
N_GENERATIONS = 50
POPULATION_SIZE = 100
P_CROSSOVER = 0.7
P_MUTATION = 0.2
TOURNAMENT_SIZE = 3
N_SELECTED_LATENT_FEATURES = 10

try:
    del creator.FitnessMax
    del creator.Individual
except:
    pass

# Create types for GA
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=LATENT_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate_individual_with_latent(individual):
    # Get selected latent features
    selected_indices = [i for i, val in enumerate(individual) if val == 1]
    
    # If too few features are selected, penalize the individual
    if len(selected_indices) < 2:
        return 0.0,
    
    # Create a modified input tensor with zeros for non-selected features
    modified_latent = train_latent[:, selected_indices]
    
    # Train a simple classifier on the selected latent features
    classifier = SVC(random_state=42)
    scores = cross_val_score(classifier, modified_latent, y_train_np, cv=5)
    accuracy = scores.mean()

    return accuracy,

toolbox.register("evaluate", evaluate_individual_with_latent)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

def run_ga():
    # Initialize population
    population = toolbox.population(n=POPULATION_SIZE)
    
    # Statistics to track progress
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Hall of fame to keep track of best individuals
    hof = tools.HallOfFame(1)
    
    # Running the algorithm
    population, logbook = algorithms.eaSimple(
        population, 
        toolbox, 
        cxpb=P_CROSSOVER, 
        mutpb=P_MUTATION, 
        ngen=N_GENERATIONS, 
        stats=stats, 
        halloffame=hof, 
        verbose=True
    )
    
    return population, logbook, hof


population, logbook, hof = run_ga()

# Get the best individual
best_individual = hof[0]
selected_latent_features = [i for i, val in enumerate(best_individual) if val == 1]
num_selected = len(selected_latent_features)

print(f"Number of selected latent features: {num_selected} out of {LATENT_FEATURES}")
print('The selected features: ', selected_latent_features)
print(f"Best fitness: {best_individual.fitness.values[0]}")


#testing the ga selected features on the models
print('Testing GA Selected Features: ')
modified_latent_train = train_latent[:, selected_latent_features]
modified_latent_test = test_latent[:, selected_latent_features]

# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(modified_latent_train, y_train_np)
svm_preds = svm_model.predict(modified_latent_test)

svm_acc = accuracy_score(y_test_np, svm_preds)
svm_percision = precision_score(y_test_np, svm_preds, average='macro')
svm_recall = recall_score(y_test_np, svm_preds, average='macro')
svm_f1 = f1_score(y_test_np, svm_preds, average='macro')

print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")
print(f"SVM Percision on latent features selected by GA: {svm_percision * 100:.4f}")
print(f"SVM Recall on latent features selected by GA: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on latent features selected by GA: {svm_f1 * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(modified_latent_train, y_train_np)
nbc_preds = nbc_model.predict(modified_latent_test)

nbc_acc = accuracy_score(y_test_np, nbc_preds)
nbc_percision = precision_score(y_test_np, nbc_preds, average='macro')
nbc_recall = recall_score(y_test_np, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test_np, nbc_preds, average='macro')

print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on latent features selected by GA: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on latent features selected by GA: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on latent features selected by GA: {nbc_f1 * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(modified_latent_train, y_train_np)
rf_preds = rf_model.predict(modified_latent_test)

rf_acc = accuracy_score(y_test_np, rf_preds)
rf_percision = precision_score(y_test_np, rf_preds, average='macro')
rf_recall = recall_score(y_test_np, rf_preds, average='macro')
rf_f1 = f1_score(y_test_np, rf_preds, average='macro')

print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by GA: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by GA: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by GA: {rf_f1 * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.2325
Epoch [2/50], Loss: 0.8518
Epoch [3/50], Loss: 0.6992
Epoch [4/50], Loss: 0.5975
Epoch [5/50], Loss: 0.4874
Epoch [6/50], Loss: 0.4330
Epoch [7/50], Loss: 0.4454
Epoch [8/50], Loss: 0.4454
Epoch [9/50], Loss: 0.4015
Epoch [10/50], Loss: 0.4143
Epoch [11/50], Loss: 0.3992
Epoch [12/50], Loss: 0.3658
Epoch [13/50], Loss: 0.3832
Epoch [14/50], Loss: 0.3801
Epoch [15/50], Loss: 0.3868
Epoch [16/50], Loss: 0.4267
Epoch [17/50], Loss: 0.3973
Epoch [18/50], Loss: 0.4244
Epoch [19/50], Loss: 0.3651
Epoch [20/50], Loss: 0.4129
Epoch [21/50], Loss: 0.4474
Epoch [22/50], Loss: 0.4228
Epoch [23/50], Loss: 0.4022
Epoch [24/50], Loss: 0.4359
Epoch [25/50], Loss: 0.3760
Epoch [26/50], Loss: 0.3486
Epoch [27/50], Loss: 0.4522
Epoch [28/50], Loss: 0.4188
Epoch [29/50], Loss: 0.4028
Epoch [30/50], Loss: 0.4168
Epoch [31/50], Loss: 0.3504
Epoch [32/50], Loss: 0.3757
Epoch [33/50], Loss: 0.3703
Epoch [34/50], Loss: 0.3896
Epoch [35/50], Loss: 0.3630


**VAE + GA**

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx')
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Variational Autoencoder-Classifier Model

class JointVAEClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointVAEClassifier, self).__init__()
        # Encoder
        self.encoder_net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128)
        )
        # Layers to produce the mean and log-variance for the latent distribution
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        # Encode input to get intermediate representation
        x_encoded = self.encoder_net(x)
        # Produce latent distribution parameters
        mu = self.fc_mu(x_encoded)
        logvar = self.fc_logvar(x_encoded)
        # Sample latent vector
        z = self.reparameterize(mu, logvar)
        # Reconstruct input and classify based on latent vector
        reconstruction = self.decoder(z)
        logits = self.classifier(z)
        return reconstruction, logits, mu, logvar


# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance reconstruction and KL loss and classification loss
lambda_recon = 0.5

def vae_combined_loss(reconstructed, original, logits, labels, mu, logvar):
    # Reconstruction loss
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    # KL divergence loss: average over batch
    loss_kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    # Classification loss
    loss_class = classification_loss_fn(logits, labels)
    # Combined loss: balance reconstruction (with KL) and classification
    return lambda_recon * (loss_recon + loss_kl) + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

# Initialize model, optimizer, and scheduler
model = JointVAEClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50


# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits, mu, logvar = model(inputs)
        loss = vae_combined_loss(reconstruction, inputs, logits, labels, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")


# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_encoded = model.encoder_net(X_train)
    train_latent = model.fc_mu(train_encoded).numpy()
    
    test_encoded = model.encoder_net(X_test)
    test_latent = model.fc_mu(test_encoded).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

# --- Genetic Algorithm Implementation ---

from deap import base, creator, tools, algorithms
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
import random
import warnings
warnings.filterwarnings("ignore")

print("Starting GA")
print("latent data:", train_latent.shape)

# Define all the needed parameters

LATENT_FEATURES = train_latent.shape[1]
N_GENERATIONS = 50
POPULATION_SIZE = 100
P_CROSSOVER = 0.7
P_MUTATION = 0.2
TOURNAMENT_SIZE = 3
N_SELECTED_LATENT_FEATURES = 10

try:
    del creator.FitnessMax
    del creator.Individual
except:
    pass

# Create types for GA
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=LATENT_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate_individual_with_latent(individual):
    # Get selected latent features
    selected_indices = [i for i, val in enumerate(individual) if val == 1]
    
    # If too few features are selected, penalize the individual
    if len(selected_indices) < 2:
        return 0.0,
    
    # Create a modified input tensor with zeros for non-selected features
    modified_latent = train_latent[:, selected_indices]
    
    # Train a simple classifier on the selected latent features
    classifier = SVC(random_state=42)
    scores = cross_val_score(classifier, modified_latent, y_train_np, cv=5)
    accuracy = scores.mean()

    return accuracy,

toolbox.register("evaluate", evaluate_individual_with_latent)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

def run_ga():
    # Initialize population
    population = toolbox.population(n=POPULATION_SIZE)
    
    # Statistics to track progress
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Hall of fame to keep track of best individuals
    hof = tools.HallOfFame(1)
    
    # Running the algorithm
    population, logbook = algorithms.eaSimple(
        population, 
        toolbox, 
        cxpb=P_CROSSOVER, 
        mutpb=P_MUTATION, 
        ngen=N_GENERATIONS, 
        stats=stats, 
        halloffame=hof, 
        verbose=True
    )
    
    return population, logbook, hof

population, logbook, hof = run_ga()

# Get the best individual
best_individual = hof[0]
selected_latent_features = [i for i, val in enumerate(best_individual) if val == 1]
num_selected = len(selected_latent_features)

print(f"Number of selected latent features: {num_selected} out of {LATENT_FEATURES}")
print('The selected features: ', selected_latent_features)
print(f"Best fitness: {best_individual.fitness.values[0]}")


#testing the ga selected features on the models
print('Testing GA Selected Features: ')
modified_latent_train = train_latent[:, selected_latent_features]
modified_latent_test = test_latent[:, selected_latent_features]

# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(modified_latent_train, y_train_np)
svm_preds = svm_model.predict(modified_latent_test)

svm_acc = accuracy_score(y_test_np, svm_preds)
svm_percision = precision_score(y_test_np, svm_preds, average='macro')
svm_recall = recall_score(y_test_np, svm_preds, average='macro')
svm_f1 = f1_score(y_test_np, svm_preds, average='macro')

print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")
print(f"SVM Percision on latent features selected by GA: {svm_percision * 100:.4f}")
print(f"SVM Recall on latent features selected by GA: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on latent features selected by GA: {svm_f1 * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(modified_latent_train, y_train_np)
nbc_preds = nbc_model.predict(modified_latent_test)

nbc_acc = accuracy_score(y_test_np, nbc_preds)
nbc_percision = precision_score(y_test_np, nbc_preds, average='macro')
nbc_recall = recall_score(y_test_np, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test_np, nbc_preds, average='macro')

print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on latent features selected by GA: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on latent features selected by GA: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on latent features selected by GA: {nbc_f1 * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(modified_latent_train, y_train_np)
rf_preds = rf_model.predict(modified_latent_test)

rf_acc = accuracy_score(y_test_np, rf_preds)
rf_percision = precision_score(y_test_np, rf_preds, average='macro')
rf_recall = recall_score(y_test_np, rf_preds, average='macro')
rf_f1 = f1_score(y_test_np, rf_preds, average='macro')

print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by GA: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by GA: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by GA: {rf_f1 * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.4941
Epoch [2/50], Loss: 1.2249
Epoch [3/50], Loss: 1.1465
Epoch [4/50], Loss: 1.0347
Epoch [5/50], Loss: 0.9915
Epoch [6/50], Loss: 0.9705
Epoch [7/50], Loss: 0.9212
Epoch [8/50], Loss: 0.8895
Epoch [9/50], Loss: 0.8770
Epoch [10/50], Loss: 0.8187
Epoch [11/50], Loss: 0.7850
Epoch [12/50], Loss: 0.8198
Epoch [13/50], Loss: 0.8275
Epoch [14/50], Loss: 0.7388
Epoch [15/50], Loss: 0.7642
Epoch [16/50], Loss: 0.7232
Epoch [17/50], Loss: 0.7016
Epoch [18/50], Loss: 0.7365
Epoch [19/50], Loss: 0.7473
Epoch [20/50], Loss: 0.7506
Epoch [21/50], Loss: 0.7083
Epoch [22/50], Loss: 0.7573
Epoch [23/50], Loss: 0.7189
Epoch [24/50], Loss: 0.7157
Epoch [25/50], Loss: 0.6991
Epoch [26/50], Loss: 0.7306
Epoch [27/50], Loss: 0.6821
Epoch [28/50], Loss: 0.6821
Epoch [29/50], Loss: 0.7513
Epoch [30/50], Loss: 0.6819
Epoch [31/50], Loss: 0.7233
Epoch [32/50], Loss: 0.6810
Epoch [33/50], Loss: 0.7190
Epoch [34/50], Loss: 0.6783
Epoch [35/50], Loss: 0.6711
